In [1]:
import os

In [2]:
pwd

'e:\\Replica\\wine_mlops\\research'

In [3]:
os.chdir('../')

In [4]:
pwd

'e:\\Replica\\wine_mlops'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str

In [6]:
from wine_quality.constants import *
from wine_quality.utils.common import read_yaml, create_directories, save_json

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath: str = CONFIG_FILE_PATH, params_filepath: str = PARAMS_FILE_PATH, schema_filepath: str = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        self.schema = read_yaml(schema_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.target_column
        )

        return model_evaluation_config

    

In [9]:
import os
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from urllib.parse import urlparse
import numpy as np
import joblib
from wine_quality.config.configuration import ConfigurationManager

In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        r2 = r2_score(actual, pred)
        mae = mean_absolute_error(actual, pred)
        mse = mean_squared_error(actual, pred)
        rmse = np.sqrt(mse)

        return r2, mae, mse, rmse
    
    def save_results(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        X_test = test_data.drop(columns=[self.config.target_column], axis=1)
        y_test = test_data[self.config.target_column]

        predicted_values = model.predict(X_test)

        (r2, mae, mse, rmse) = self.eval_metrics(y_test, predicted_values)

        #Saving the metrices in a json file

        metrics = {"rmse": rmse, "r2": r2, "mae": mae, "mse": mse}
        save_json(path = Path(self.config.metric_file_name), data = metrics)

In [13]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.save_results()

except Exception as e:
    raise e


[2025-05-01 09:16:29,159: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-05-01 09:16:29,161: INFO: common: yaml file: params.yaml loaded successfully]
[2025-05-01 09:16:29,164: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-05-01 09:16:29,166: INFO: common: created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_model_evaluation_config'